# `trained_model_adapter.py` — Website-Compatible Prediction Adapter

## Purpose
Wraps the trained `RoBERTa-GCN` model (via `SentimentPredictor` from `inference.py`) to provide the **exact JSON response format the website backend expects** from its `/predict` endpoint.

## Key Features
- **3-class sentiment**: Outputs `negative / neutral / positive` (no binary).
- **Aspect mention detection**: Uses keyword matching to check if an aspect is actually discussed in the review before running model inference on it. Unmentioned aspects get `'not_mentioned'` instead of a potentially misleading prediction.
- **Conflict score**: Computes a real mixed-sentiment conflict probability using geometric mean of the confident-positive and confident-negative aspect scores.

## Usage
```python
adapter = TrainedModelAdapter('/path/to/best_model.pt', temperature=0.5)
result  = adapter.predict('The colour is brilliant but the smell is horrible.')
# result: {aspects: [...], conflict_prob: float, timings: {...}}
```

In [ ]:
import sys, os
import numpy as np
from typing import Dict, List

# ── Path setup ─────────────────────────────────────────────────────────────
ML_RESEARCH   = os.path.dirname(os.path.abspath(''))  # ml-research/
inference_dir = os.path.join(ML_RESEARCH, 'outputs', 'cosmetic_sentiment_v1', 'evaluation')
src_dir       = os.path.join(ML_RESEARCH, 'src')
for _d in [inference_dir, src_dir]:
    if _d not in sys.path: sys.path.insert(0, _d)

from inference import SentimentPredictor  # The actual model wrapper

## Aspect Keyword Map
Each aspect has a set of keywords. If **none** appear in the review, the aspect is marked `'not_mentioned'` and no inference is run — this saves compute and avoids misleading the user.

In [ ]:
LABEL_NAMES = ['negative', 'neutral', 'positive']  # Match model's training label order

# Simple substring-matching keyword lists — fast and sufficient for English reviews
ASPECT_KEYWORDS: Dict[str, List[str]] = {
    'colour':       ['colour', 'color', 'shade', 'hue', 'pigment', 'tint', 'bright', 'dark', 'vibrant', 'rich', 'tone', 'dye', 'red', 'pink', 'nude', 'bold'],
    'smell':        ['smell', 'scent', 'fragrance', 'odor', 'odour', 'aroma', 'perfume', 'stink', 'reek', 'chemical', 'fresh'],
    'texture':      ['texture', 'feel', 'consistency', 'thick', 'thin', 'smooth', 'rough', 'creamy', 'gritty', 'silky', 'lumpy', 'buttery', 'sticky', 'waxy'],
    'price':        ['price', 'cost', 'expensive', 'cheap', 'afford', 'worth', 'value', 'money', 'pricey', 'overpriced', 'budget', 'high', 'low', 'deal', 'pay'],
    'stayingpower': ['stay', 'last', 'lasting', 'long', 'hour', 'fade', 'wear', 'smear', 'transfer', 'hold', 'all day', 'evening', 'crumble'],
    'shipping':     ['ship', 'deliver', 'delivery', 'arrived', 'arrive', 'package', 'fast', 'slow', 'late', 'quick', 'days', 'courier', 'dispatch'],
    'packing':      ['pack', 'packaging', 'box', 'container', 'tube', 'bottle', 'wrap', 'seal', 'cap', 'lid', 'compact', 'broken', 'damaged', 'intact'],
}


def _is_mentioned(text: str, aspect: str) -> bool:
    """Return True if any keyword for this aspect appears in the text (case-insensitive)."""
    text_lower = text.lower()
    return any(kw in text_lower for kw in ASPECT_KEYWORDS.get(aspect, []))

## Conflict Score
Only considers **mentioned** aspects. A review is 'conflicted' when it has at least one confidently-positive AND one confidently-negative mentioned aspect simultaneously.

Score = geometric mean of `max_pos_confidence` × `max_neg_confidence` — high only when BOTH sides are confident.

In [ ]:
def _compute_conflict_score(aspects_result: List[dict]) -> float:
    CONF_THRESHOLD = 0.45   # Min confidence to count as 'committed' to a sentiment

    mentioned = [a for a in aspects_result if a['label'] != 'not_mentioned']  # Ignore absent aspects

    pos_confs = [a['confidence'] for a in mentioned if a['label'] == 'positive' and a['confidence'] >= CONF_THRESHOLD]
    neg_confs = [a['confidence'] for a in mentioned if a['label'] == 'negative' and a['confidence'] >= CONF_THRESHOLD]

    if not pos_confs or not neg_confs:
        # Soft score: if both polarities appear with low confidence, mark mild conflict
        labels = [a['label'] for a in mentioned]
        if 'positive' in labels and 'negative' in labels: return 0.30
        return 0.0

    # Strong conflict: geometric mean ensures BOTH sides must be confident for a high score
    return min(float(np.sqrt(max(pos_confs) * max(neg_confs))), 1.0)

## TrainedModelAdapter Class

In [ ]:
class TrainedModelAdapter:
    """
    Wraps the trained SentimentPredictor with website-compatible predict() signature.
    """
    def __init__(self, checkpoint_path: str, temperature: float = 0.5):
        """
        Args:
            checkpoint_path: Absolute path to best_model.pt
            temperature: Softmax temperature for calibration (< 1.0 sharpens predictions)
        """
        self.predictor     = SentimentPredictor(checkpoint_path, temperature=temperature)
        self.aspect_names  = self.predictor.aspect_names  # List like ['colour', 'smell', ...]
        self.temperature   = temperature

    @property
    def device(self):
        return self.predictor.device

    def predict(self, text: str, enable_msr: bool = True) -> Dict:
        """
        Run prediction across all aspects.

        Returns a dict compatible with the website /predict endpoint:
        {
            'aspects': [{'name', 'label', 'confidence', 'probs', 'mentioned', ...}, ...],
            'conflict_prob': float,
            'timings': {'total_ms': float}
        }
        """
        import time
        t0 = time.time()
        aspects_result = []

        for asp_name in self.aspect_names:
            if not _is_mentioned(text, asp_name):
                # Skip model inference for unmentioned aspects — save compute + avoid misleading output
                aspects_result.append({
                    'name': asp_name, 'label': 'not_mentioned', 'confidence': 0.0,
                    'probs': [0.0, 0.0, 0.0], 'before': {'label': 'not_mentioned', 'confidence': 0.0},
                    'after': {'label': 'not_mentioned', 'confidence': 0.0},
                    'changed_by_msr': False, 'top_tokens': [], 'mentioned': False,
                })
                continue

            raw  = self.predictor.predict(text, asp_name, return_attention=True)  # Run model inference
            neg, neu, pos = raw['probabilities']['negative'], raw['probabilities']['neutral'], raw['probabilities']['positive']

            aspects_result.append({
                'name'         : asp_name,
                'label'        : raw['sentiment'],        # 'negative' | 'neutral' | 'positive'
                'confidence'   : raw['confidence'],       # max probability after temperature scaling
                'probs'        : [neg, neu, pos],         # [neg, neu, pos] probabilities
                'before'       : {'label': raw['sentiment'], 'confidence': raw['confidence']},
                'after'        : {'label': raw['sentiment'], 'confidence': raw['confidence']},
                'changed_by_msr': False,
                'top_tokens'   : raw.get('top_tokens', []),
                'mentioned'    : True,
            })

        conflict_prob = _compute_conflict_score(aspects_result)  # Real conflict score
        elapsed_ms    = (time.time() - t0) * 1000

        return {
            'aspects'      : aspects_result,
            'conflict_prob': conflict_prob,
            'timings'      : {'total_ms': elapsed_ms},
        }

## Quick Test

In [ ]:
if __name__ == '__main__':
    ckpt    = os.path.join(project_root, 'ml-research', 'outputs', 'cosmetic_sentiment_v1', 'best_model.pt')
    adapter = TrainedModelAdapter(ckpt)

    test_reviews = [
        "This foundation is amazing! Great colour, stays all day, smells wonderful and the price is perfect.",
        "Worst lipstick ever. Awful colour, fades fast, chemical smell, packaging broke, way overpriced.",
        "Love the colour and texture, but the smell is absolutely horrible.",
    ]

    for review in test_reviews:
        result = adapter.predict(review)
        print(f'\nReview: {review[:80]}')
        for asp in result['aspects']:
            if asp['mentioned']:
                print(f"  {asp['name']:14s}: {asp['label']:8s}  conf={asp['confidence']:.3f}")
        print(f"  conflict_prob = {result['conflict_prob']:.3f}")